# Lección 1 · Tu primer modelo de optimización

**Qué vas a aprender**
- Qué es la investigación de operaciones (IO).
- Las 3 piezas de todo modelo: decisiones, objetivo y restricciones.
- Por qué la mejor solución está en una esquina.
- Escribir el modelo en Pyomo pieza por pieza, imprimiendo cada pieza.
- Leer la solución: qué recurso es el cuello de botella.

**Cómo usar este notebook**
- Corre las celdas en orden, una por una, con `Shift + Enter`.
- Cada celda hace una sola cosa e imprime lo que creó.
- Antes de correr una celda, adivina qué va a imprimir. Si fallas, ahí está el aprendizaje.

## 0. Preparar el entorno
Esta celda instala dos cosas:
- **Pyomo**: el lenguaje para escribir el modelo.
- **HiGHS**: el *solver*, el programa que encuentra la mejor solución.

En Colab tarda unos 20 segundos. En tu computador basta correrla una vez.

In [ ]:
%pip install -q pyomo highspy

## 1. ¿Qué es la investigación de operaciones?
Es usar matemáticas para tomar **la mejor decisión cuando los recursos son limitados**.

Todo problema de IO responde 3 preguntas:
1. **¿Qué decido?** → las *variables de decisión*.
2. **¿Qué quiero lograr?** → la *función objetivo* (maximizar o minimizar).
3. **¿Qué me limita?** → las *restricciones*.

**Analogía.** Planear tu semana. Decides cuántas horas das al trabajo, al polaco y a tu hijo (variables).
Quieres el mayor bienestar (objetivo). Pero la semana tiene 168 horas y hay que dormir (restricciones).

## 2. El problema del taller
Un taller de equipos para la industria de energía fabrica dos productos: **válvulas** y **conectores**.

| | Válvula | Conector | Disponible por semana |
|---|---|---|---|
| Horas de mecanizado por unidad | 2 | 1 | 100 h |
| Horas de prueba de presión por unidad | 1 | 1 | 80 h |
| Ganancia por unidad | 300 € | 200 € | |
| Máximo que compra el mercado | 40 | sin límite | |

**Pregunta:** ¿cuántas válvulas y cuántos conectores fabricar por semana para ganar lo máximo?

Las 3 piezas:
- **Decido:** V = válvulas, C = conectores.
- **Objetivo:** maximizar 300·V + 200·C.
- **Restricciones:**
  - mecanizado: 2·V + C ≤ 100
  - prueba: V + C ≤ 80
  - mercado: V ≤ 40
  - no se fabrica en negativo: V ≥ 0, C ≥ 0

Esto es un **programa lineal (LP)**: solo hay sumas de variables multiplicadas por números.
No hay V², ni V·C, ni "si pasa esto, entonces…".

## 3. Primero, a mano: probar planes
Antes de usar un solver, probemos planes con Python normal. Así ves qué hace el solver por dentro.
Un plan es **factible** si cumple todas las restricciones.

In [ ]:
def eur(x):
    return f"{x:,.0f}".replace(",", ".") + " €"


def ganancia(v, c):
    return 300 * v + 200 * c


def revisar(v, c):
    # Imprime la ganancia del plan y qué restricciones no cumple
    cumple = {
        "mecanizado (≤ 100 h)": 2 * v + c <= 100,
        "prueba (≤ 80 h)": v + c <= 80,
        "mercado (≤ 40 válvulas)": v <= 40,
    }
    factible = all(cumple.values())
    print(f"V={v:>3}, C={c:>3} → ganancia {eur(ganancia(v, c)):>9} | factible: {factible}")
    for nombre, ok in cumple.items():
        if not ok:
            print(f"      ✗ no cumple {nombre}")


revisar(40, 0)   # solo válvulas
revisar(0, 80)   # solo conectores
revisar(40, 40)  # mucho de los dos
revisar(30, 40)

**Tu turno.** Prueba otros planes en la celda de abajo. ¿Encuentras uno que gane más de 17.000 €?

In [ ]:
revisar(25, 50)

## 4. Fuerza bruta: probarlos todos
Con 2 productos podemos probar todos los planes enteros de 0 a 100 unidades: 101 × 101 = 10.201 planes.

In [ ]:
mejor, probados = None, 0
for v in range(101):
    for c in range(101):
        probados += 1
        factible = 2 * v + c <= 100 and v + c <= 80 and v <= 40
        if factible and (mejor is None or ganancia(v, c) > ganancia(*mejor)):
            mejor = (v, c)

print("Planes probados:", f"{probados:,}".replace(",", "."))
print(f"Mejor plan: V={mejor[0]}, C={mejor[1]} → {eur(ganancia(*mejor))}")

**¿Por qué no hacer siempre esto?** Con 10 productos y hasta 1.000 unidades de cada uno, habría
1.001¹⁰ ≈ 10³⁰ planes. Probando mil millones por segundo, tardarías unas 2.000 veces la edad del universo.

El solver llega al mejor plan **sin probarlos todos**. La siguiente sección muestra el truco.

## 5. Dibujarlo: la región factible y sus esquinas
- Cada restricción es una línea.
- Los planes que cumplen todas forman una zona: la **región factible** (sombreada).
- La línea punteada une planes con la misma ganancia. Si la empujas hacia arriba y a la derecha,
  el último punto que toca la región es el óptimo.
- Ese punto siempre es una **esquina**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

v = np.linspace(0, 60, 200)
esquinas = [(0, 0), (40, 0), (40, 20), (20, 60), (0, 80)]

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(v, 100 - 2 * v, label="mecanizado: 2V + C = 100")
ax.plot(v, 80 - v, label="prueba: V + C = 80")
ax.axvline(40, color="gray", label="mercado: V = 40")
ax.fill([e[0] for e in esquinas], [e[1] for e in esquinas], alpha=0.25, label="región factible")
ax.plot(v, (18000 - 300 * v) / 200, "k--", label="ganancia = 18.000 €")
for e in esquinas:
    ax.plot(*e, "ko")
    ax.annotate(f"{e}\n{eur(ganancia(*e))}", e, textcoords="offset points", xytext=(6, 4), fontsize=9)
ax.set(xlim=(0, 60), ylim=(0, 110), xlabel="V (válvulas)", ylabel="C (conectores)")
ax.legend(loc="upper right")
plt.show()

In [ ]:
for e in esquinas:
    print(f"Esquina {e}: {eur(ganancia(*e))}")

La mejor esquina es (20, 60) con 18.000 €: lo mismo que encontró la fuerza bruta,
pero revisando 5 puntos en vez de 10.201.

Así trabaja el **método simplex**: salta de esquina en esquina, siempre a una mejor,
hasta que ninguna vecina mejora.

## 6. El mismo modelo en Pyomo, pieza por pieza
Pyomo es una librería de Python para escribir modelos. Se parece a escribir las fórmulas a mano.
Vamos a crear cada pieza e imprimirla.

In [ ]:
import pyomo.environ as pyo

m = pyo.ConcreteModel(name="taller")
print("Modelo:", m.name)
print("Componentes:", [c.name for c in m.component_objects()])

### 6.1 Conjuntos (*sets*): las listas de cosas
Aquí hay dos listas: productos y máquinas. Todo lo demás se indexa con ellas.

In [ ]:
m.PRODUCTOS = pyo.Set(initialize=["valvula", "conector"])
m.MAQUINAS = pyo.Set(initialize=["mecanizado", "prueba"])

print("PRODUCTOS:", list(m.PRODUCTOS))
print("MAQUINAS:", list(m.MAQUINAS))

### 6.2 Parámetros (*params*): los datos fijos
El solver los lee, pero no los cambia. Fíjate en `Index`: dice con qué conjunto se indexa cada parámetro.

In [ ]:
m.ganancia = pyo.Param(m.PRODUCTOS, initialize={"valvula": 300, "conector": 200})
m.horas = pyo.Param(m.PRODUCTOS, m.MAQUINAS, initialize={
    ("valvula", "mecanizado"): 2, ("valvula", "prueba"): 1,
    ("conector", "mecanizado"): 1, ("conector", "prueba"): 1,
})
m.capacidad = pyo.Param(m.MAQUINAS, initialize={"mecanizado": 100, "prueba": 80})
m.max_valvulas = pyo.Param(initialize=40)

m.ganancia.pprint()
m.horas.pprint()
m.capacidad.pprint()
m.max_valvulas.pprint()

### 6.3 Variables (*vars*): lo que decide el solver
Antes de resolver no tienen valor: `Value` dice `None`. `Lower` es 0 porque no se fabrica en negativo.

In [ ]:
m.x = pyo.Var(m.PRODUCTOS, within=pyo.NonNegativeReals)
m.x.pprint()

### 6.4 Función objetivo: lo que queremos maximizar
`sum(... for p in m.PRODUCTOS)` escribe la suma una sola vez para todos los productos. Mira cómo Pyomo la expande.

In [ ]:
m.obj = pyo.Objective(
    expr=sum(m.ganancia[p] * m.x[p] for p in m.PRODUCTOS),
    sense=pyo.maximize,
)
print("Objetivo:", m.obj.expr)

### 6.5 Restricciones (*constraints*): lo que nos limita
Una **regla** escrita una vez se aplica a cada máquina: Pyomo crea una restricción por máquina.

In [ ]:
def regla_capacidad(m, maq):
    return sum(m.horas[p, maq] * m.x[p] for p in m.PRODUCTOS) <= m.capacidad[maq]


m.cap = pyo.Constraint(m.MAQUINAS, rule=regla_capacidad)
m.mercado = pyo.Constraint(expr=m.x["valvula"] <= m.max_valvulas)

for maq in m.MAQUINAS:
    print(f"cap[{maq}]:", m.cap[maq].expr)
print("mercado:", m.mercado.expr)

### 6.6 Todo junto
`pprint()` muestra el modelo completo. Compáralo con las fórmulas de la sección 2.

In [ ]:
m.pprint()

## 7. Resolver
Le pasamos el modelo al solver HiGHS.

In [ ]:
solver = pyo.SolverFactory("appsi_highs")
resultado = solver.solve(m)
print("Estado:", resultado.solver.termination_condition)

Estados que vas a ver:
- **optimal**: encontró el mejor plan y lo puede demostrar.
- **infeasible**: ningún plan cumple todas las restricciones.
- **unbounded**: la ganancia puede crecer sin límite. Casi siempre falta una restricción.

In [ ]:
for p in m.PRODUCTOS:
    print(f"{p}: {pyo.value(m.x[p]):.0f} unidades")
print("Ganancia:", eur(pyo.value(m.obj)))

## 8. Leer la solución: ¿qué nos limita?
Para cada restricción miramos cuánto usamos y cuánto sobra.
- Lo que sobra se llama **holgura** (*slack*).
- Una restricción con holgura 0 está **activa** (*binding*): es un cuello de botella.

In [ ]:
filas = [(f"cap[{maq}]", m.cap[maq]) for maq in m.MAQUINAS] + [("mercado", m.mercado)]
for nombre, restriccion in filas:
    usado = pyo.value(restriccion.body)
    limite = pyo.value(restriccion.upper)
    print(f"{nombre:<16} usado {usado:>4.0f} de {limite:>4.0f} → holgura {limite - usado:>3.0f}")

**Lectura para el jefe del taller:**
- Mecanizado y prueba están al 100 %: son los cuellos de botella.
- El límite del mercado no molesta hoy: sobran 20 válvulas de demanda.
- Para ganar más hacen falta más horas de máquina, no más clientes. En la lección 2 calculamos cuánto vale cada hora extra.

## 9. ¿Qué pasa si…? Empaquetar el modelo en una función
Ya viste cada pieza. Ahora la metemos en una función para cambiar datos y resolver de nuevo.
Así se trabaja en la práctica: primero pieza por pieza para entender; después, una función para experimentar.

In [ ]:
def resolver_taller(ganancia, horas, capacidad, max_valvulas=40):
    m = pyo.ConcreteModel()
    m.P = pyo.Set(initialize=list(ganancia))
    m.M = pyo.Set(initialize=list(capacidad))
    m.x = pyo.Var(m.P, within=pyo.NonNegativeReals)
    m.obj = pyo.Objective(expr=sum(ganancia[p] * m.x[p] for p in m.P), sense=pyo.maximize)
    m.cap = pyo.Constraint(m.M, rule=lambda m, k: sum(horas.get((p, k), 0) * m.x[p] for p in m.P) <= capacidad[k])
    m.mercado = pyo.Constraint(expr=m.x["valvula"] <= max_valvulas)
    pyo.SolverFactory("appsi_highs").solve(m)
    plan = {p: round(pyo.value(m.x[p]), 1) for p in m.P}
    return plan, eur(pyo.value(m.obj))


ganancia_base = {"valvula": 300, "conector": 200}
horas_base = {("valvula", "mecanizado"): 2, ("valvula", "prueba"): 1,
              ("conector", "mecanizado"): 1, ("conector", "prueba"): 1}
capacidad_base = {"mecanizado": 100, "prueba": 80}

print("Base:", resolver_taller(ganancia_base, horas_base, capacidad_base))

In [ ]:
print("Mecanizado 101 h:", resolver_taller(ganancia_base, horas_base, {"mecanizado": 101, "prueba": 80}))

In [ ]:
print("Conector a 350 €:", resolver_taller({"valvula": 300, "conector": 350}, horas_base, capacidad_base))

In [ ]:
print("Mercado de 10 válvulas:", resolver_taller(ganancia_base, horas_base, capacidad_base, max_valvulas=10))

**Qué aprendiste al cambiar datos:**
1. Una hora más de mecanizado sube la ganancia 100 €. Ese número tiene nombre: **precio sombra**. Es el tema de la lección 2.
2. Si el conector sube a 350 €, el óptimo salta a otra esquina: solo conectores.
3. Si el mercado compra solo 10 válvulas, el cuello de botella cambia: ahora limita el mercado y sobran 10 h de mecanizado.

## 10. Resumen: conceptos de hoy
| Concepto | Qué es | En el taller | En inglés |
|---|---|---|---|
| Variable de decisión | Lo que el modelo decide | V, C | decision variable |
| Función objetivo | Lo que se maximiza o minimiza | 300·V + 200·C | objective function |
| Restricción | Un límite que el plan debe cumplir | 2·V + C ≤ 100 | constraint |
| Región factible | Todos los planes que cumplen todo | el polígono sombreado | feasible region |
| Solución óptima | El mejor plan factible | V = 20, C = 60: 18.000 € | optimal solution |
| Restricción activa | Se usa al 100 %: cuello de botella | mecanizado y prueba | binding constraint |
| Holgura | Lo que sobra de un recurso | mercado: 20 válvulas | slack |
| Programa lineal | Modelo con solo sumas de variables × números | este modelo | linear program (LP) |
| Solver | Programa que encuentra el óptimo | HiGHS | solver |

## 11. Ejercicios: el modelo crece
Usa `resolver_taller`: solo cambias datos.
1. **Tercer producto.** Agrega la **brida**: 1 h de mecanizado, 2 h de prueba, ganancia 250 €. ¿Cambia el plan? ¿Por qué?
2. **Mantenimiento.** La prueba de presión baja a 70 h esta semana. ¿Cuánto dinero se pierde?
3. **Quitar restricciones.** ¿Qué pasa si no existe el límite de mecanizado? ¿Y si no existe ninguna restricción?

In [ ]:
# Escribe aquí tus intentos

## En la entrevista
Cuando te den un problema, dilo en este orden:
> *"First, the decisions. Second, the objective. Third, the constraints. After solving, I check which constraints are binding: those are the bottlenecks."*

## Soluciones (no mires antes de intentar)

In [ ]:
# Ejercicio 1: la brida
ganancia_3 = {**ganancia_base, "brida": 250}
horas_3 = {**horas_base, ("brida", "mecanizado"): 1, ("brida", "prueba"): 2}
print(resolver_taller(ganancia_3, horas_3, capacidad_base))

La brida no entra. Usa 1 h de mecanizado y 2 h de prueba, y cada una de esas horas vale 100 €
(lo viste con el mecanizado en la sección 9). Fabricar una brida "cuesta" 1×100 + 2×100 = 300 €
de ganancia que dejas de hacer con los otros productos, y solo deja 250 €.
En la lección 2 este número se llama **costo reducido**.

In [ ]:
# Ejercicio 2: prueba de presión con 70 h
print(resolver_taller(ganancia_base, horas_base, {"mecanizado": 100, "prueba": 70}))

Ganancia 17.000 €: se pierden 1.000 €. Son 10 horas × 100 € por hora, el precio sombra de la prueba.

In [ ]:
# Ejercicio 3a: sin límite de mecanizado
print(resolver_taller(ganancia_base, horas_base, {"prueba": 80}))

# Ejercicio 3b: sin ninguna restricción.
# load_solutions=False evita el error de cargar una solución que no existe.
m3 = pyo.ConcreteModel()
m3.x = pyo.Var(["valvula", "conector"], within=pyo.NonNegativeReals)
m3.obj = pyo.Objective(expr=300 * m3.x["valvula"] + 200 * m3.x["conector"], sense=pyo.maximize)
resultado3 = pyo.SolverFactory("appsi_highs").solve(m3, load_solutions=False)
print("Sin restricciones:", resultado3.solver.termination_condition)

Sin mecanizado, limitan la prueba (80 h) y el mercado (40 válvulas): 40 válvulas + 40 conectores, 20.000 €.
Sin ninguna restricción, el modelo es *unbounded*: podrías fabricar infinito.
En la vida real eso casi siempre significa que olvidaste una restricción.